# Lekcija 31: Osnove neuronskih mreža

U lekciji 30 ručno smo odabrali smjer projekcije $w$ (razliku srednjih vrijednosti klasa) za razdvajanje dviju klasa. To je uspjelo, ali je zahtijevalo da čovjek uoči dobru heuristiku. Ova lekcija zamjenjuje čovjeka **gradijentnim spustom**: automatskim postupkom koji *uči* $w$ i $b$ izravno iz podataka, uzastopno ih malo pomičući kako bi smanjio gubitak. Postupak &mdash; prolaz unaprijed, gubitak, prolaz unatrag, ažuriranje &mdash; cijela je petlja treniranja svake neuronske mreže u ovom kolegiju, bez obzira na njezinu veličinu.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## Nova biblioteka i nova struktura podataka: PyTorch i tenzori

Svaka dosadašnja lekcija uvozila je `cv2` za operacije nad slikama. Ova lekcija umjesto toga uvozi `torch` &mdash; prvo pojavljivanje PyTorcha u kolegiju, a odsad će se većina lekcija oslanjati na njega umjesto na OpenCV. Razlog je što izgradnja i treniranje neuronskih mreža zahtijevaju dvije stvari za koje OpenCV nije bio zamišljen: automatsku diferencijaciju (računanje gradijenata proizvoljno složenih funkcija, što koristimo u nastavku) i GPU ubrzanje velikih množenja matrica od kojih su neuronske mreže izgrađene.

S PyTorchom dolazi nova riječ: **tenzor**. Tenzor je samo opći naziv za mrežu brojeva proizvoljne dimenzionalnosti: skalar je 0-dimenzijski tenzor, vektor 1-dimenzijski, matrica 2-dimenzijski, a skup matrica &mdash; primjerice serija slika &mdash; 4-dimenzijski tenzor. Svako dosad korišteno polje u ovom kolegiju &mdash; NumPy polja, slike, matrice težina &mdash; zapravo je cijelo vrijeme bilo tenzor; PyTorchov `torch.Tensor` jednostavno je NumPyjev `ndarray` s dva dodatka: može pratiti `.grad` za automatsku diferencijaciju i može se nalaziti na GPU-u umjesto na CPU-u radi brzog, paralelnog računanja. Tenzorov `.shape` govori upravo ono što bi NumPyjev `.shape` govorio za isto polje &mdash; te su dvije biblioteke dovoljno slične da je pretvorba između njih (`torch.tensor(numpy_array)`, `tensor.numpy()`) praktički besplatna.

Tenzori slika donose dvije konvencije koje treba odmah istaknuti:

- **Kanali na prvom mjestu.** PyTorch pohranjuje slike u boji kao `(channel, height, width)`, suprotno NumPy/OpenCV rasporedu `(height, width, channel)` iz lekcije 1 &mdash; pretvorba zahtijeva pomicanje osi kanala, a ne samo promjenu oblika: `image.transpose(2, 0, 1)` za NumPy polje (HWC&rarr;CHW) ili `tensor.permute(2, 0, 1)` za tenzor.
- **Obvezna dimenzija serije.** PyTorchovi slojevi i funkcije gotovo uvijek očekuju os serije na prvom mjestu, čak i za samo jedan primjer: jedna slika ulazi u obliku `(1, channel, height, width)`, a ne `(channel, height, width)`. Zaboravljanje te osi čest je uzrok pogrešaka nepodudaranja oblika; `tensor.unsqueeze(0)` umeće os serije veličine 1 na početak, a `tensor.squeeze(0)` ponovno je uklanja kada se vratite obradi jednog rezultata.

## Problem binarne klasifikacije

Ponovno koristimo isti skup podataka s dvjema klasama iz lekcije 30, kako bismo naučeni smjer u nastavku mogli izravno usporediti s ručno odabranim.

In [ ]:
rng = np.random.default_rng(1)
class_a = rng.normal(loc=[-2, -1], scale=0.8, size=(60, 2))
class_b = rng.normal(loc=[2, 1.5], scale=0.8, size=(60, 2))
X = np.vstack([class_a, class_b])
y = np.concatenate([np.zeros(60), np.ones(60)])  # class A = 0, class B = 1

plt.scatter(*class_a.T, s=15, label='class A (y=0)')
plt.scatter(*class_b.T, s=15, label='class B (y=1)')
plt.legend(fontsize=8)
plt.gca().set_aspect('equal')
plt.title('Same two-class dataset as Lesson 30')
plt.show()

## Od sirove vrijednosti do vjerojatnosti

Prisjetite se projekcije iz lekcije 30: jedan neuron računa sirovu vrijednost $z = w^\top x + b$ iz ulaza $x$, a zatim klasificira prema njezinu predznaku. Cilj treniranja jest pronaći $w$ i $b$ koji daju pozitivan $z$ za svaku točku klase B i negativan za svaku točku klase A &mdash; ali umjesto ručnog odabira kao u lekciji 30, želimo da ih gradijentni spust pronađe automatski.

Problem je što je klasifikacijska točnost (točno ili pogrešno) gotovo svugdje ravna i skokovito se mijenja na granici odluke &mdash; nema korisnog gradijenta koji bismo slijedili, pa gradijentni spust nema niz što silaziti. Rješenje je prestati tražiti strogi odgovor da/ne i umjesto toga sirovu vrijednost provući kroz **sigmoidnu** funkciju, koja svaki realni broj preslikava u glatku vrijednost između 0 i 1 &mdash; što čitamo kao „koliko je model siguran da ova točka pripada klasi B?”

$$\sigma(z) = \frac{1}{1+e^{-z}}$$

Veliki pozitivni $z$ preslikava se blizu 1, veliki negativni $z$ blizu 0, a $z=0$ točno u 0.5, jednako između obiju mogućnosti.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

zs = np.linspace(-6, 6, 200)
plt.plot(zs, sigmoid(zs))
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.axvline(0, color='gray', linestyle='--', linewidth=1)
plt.xlabel('z = w.x + b')
plt.ylabel('sigma(z)')
plt.title('The sigmoid activation')
plt.show()

## Binarna unakrsna entropija: pretvaranje vjerojatnosti u gubitak za treniranje

Sama vjerojatnost još nije veličina koju možemo optimizirati. Treniranje zahtijeva jedan broj &mdash; nazvan **pogreška** ili **gubitak** &mdash; koji je malen kada su vjerojatnosti modela dobre, a velik kada su loše, usrednjen preko cijelog skupa podataka. **Binarna unakrsna entropija** gradi taj broj kažnjavajući svako predviđanje prema njegovoj sigurnosti i tome je li ta sigurnost opravdana: ako je stvarna oznaka 1, kazna je $-\log(p)$ (blizu nule kada je $p$ blizu 1, a raste prema beskonačnosti kada $p\to 0$ &mdash; samouvjereno pogrešno predviđanje strogo se kažnjava); ako je stvarna oznaka 0, dobivamo zrcalni slučaj, $-\log(1-p)$.

In [ ]:
p_range = np.linspace(0.001, 0.999, 200)
loss_if_true_1 = -np.log(p_range)
loss_if_true_0 = -np.log(1 - p_range)

plt.plot(p_range, loss_if_true_1, label='true label = 1:  -log(p)')
plt.plot(p_range, loss_if_true_0, label='true label = 0:  -log(1-p)')
plt.xlabel('predicted probability p')
plt.ylabel('per-example loss')
plt.legend(fontsize=8)
plt.title('Binary cross-entropy: the penalty for one prediction')
plt.show()

Usrednjavanje te kazne po primjeru preko cijelog skupa podataka daje ukupni gubitak:

$$L = -\frac{1}{n}\sum_i \big[y_i \log \sigma(z_i) + (1-y_i)\log(1-\sigma(z_i))\big]$$

To su upravo dvije gornje krivulje, odabrane pomoću $y_i$ (samo je jedan od dvaju članova ikad različit od nule jer je $y_i$ jednak 0 ili 1) i usrednjene preko svih točaka. Budući da su i sigmoidna funkcija i logaritam glatki, $L$ svugdje ima dobro definiran gradijent &mdash; gubitak se glatko smanjuje kako predviđanja postaju točnija, umjesto da se mijenja samo na granici odluke.

## Povratno širenje pogreške: primjena lančanog pravila

„Backprop” je samo lančano pravilo iz lekcija 12–13 primijenjeno na kompoziciju funkcija: $L$ ovisi o $\sigma(z)$, koji ovisi o $z=w^\top x + b$, koji ovisi o $w$ i $b$. Kretanje unatrag tim lancem (preskačemo algebru, koja je standardno, ali pomalo zamorno pojednostavnjenje) daje iznenađujuće jednostavan rezultat:

$$\frac{\partial L}{\partial z_i} = \sigma(z_i) - y_i, \qquad \frac{\partial L}{\partial w} = \frac{1}{n}X^\top(\sigma(z)-y), \qquad \frac{\partial L}{\partial b} = \frac{1}{n}\sum_i(\sigma(z_i)-y_i)$$

Riječima: gradijent je samo usrednjena *pogreška predviđanja* (ponderirana s $x$ za gradijent $w$). Izrazito pogrešna predviđanja snažno pomiču težine; točna ih gotovo uopće ne mijenjaju.

In [ ]:
def forward(X, w, b):
    z = X @ w + b
    return sigmoid(z), z

def bce_loss(p, y, eps=1e-9):
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def backward(X, p, y):
    grad_z = (p - y) / len(y)
    grad_w = X.T @ grad_z
    grad_b = grad_z.sum()
    return grad_w, grad_b

### Provjera ispravnosti: provjera gradijenta

Prije nego što vjerujemo ručno izvedenom gradijentu, standardna je praksa numerički ga provjeriti: svaki parametar malo promijeniti, vidjeti koliko se gubitak doista promijenio i zatim usporediti s analitičkom formulom.

In [ ]:
w_test, b_test = np.array([0.5, -0.3]), 0.2
p_test, _ = forward(X, w_test, b_test)
grad_w_analytic, grad_b_analytic = backward(X, p_test, y)

def loss_at(w, b):
    p, _ = forward(X, w, b)
    return bce_loss(p, y)

eps = 1e-5
grad_w_numeric = np.zeros(2)
for i in range(2):
    w_plus, w_minus = w_test.copy(), w_test.copy()
    w_plus[i] += eps
    w_minus[i] -= eps
    grad_w_numeric[i] = (loss_at(w_plus, b_test) - loss_at(w_minus, b_test)) / (2 * eps)
grad_b_numeric = (loss_at(w_test, b_test + eps) - loss_at(w_test, b_test - eps)) / (2 * eps)

print(f'analytic grad_w: {grad_w_analytic}, numeric: {grad_w_numeric}')
print(f'analytic grad_b: {grad_b_analytic:.6f}, numeric: {grad_b_numeric:.6f}')
print(f'max discrepancy: {max(np.abs(grad_w_analytic - grad_w_numeric).max(), abs(grad_b_analytic - grad_b_numeric)):.2e}')

### Provjera ispravnosti: PyTorch autograd

Kao drugu, neovisnu provjeru, dopuštamo PyTorchu da automatski izračuna isti gradijent pomoću `.backward()`, koristeći ugrađeni `binary_cross_entropy_with_logits` (koji kombinira sigmoidnu funkciju i gubitak u jednoj numerički stabilnoj funkciji).

In [ ]:
X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)
w_t = torch.tensor(w_test, dtype=torch.float64, requires_grad=True)
b_t = torch.tensor(b_test, dtype=torch.float64, requires_grad=True)

z_t = X_t @ w_t + b_t
loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z_t, y_t)
loss_t.backward()

print(f'our analytic grad_w: {grad_w_analytic}')
print(f'torch autograd grad_w: {w_t.grad.numpy()}')
print(f'max discrepancy: {np.abs(grad_w_analytic - w_t.grad.numpy()).max():.2e}')

Dvije neovisne provjere &mdash; numeričke konačne razlike i PyTorchova automatska diferencijacija &mdash; slažu se s ručno izvedenom formulom na mnogo decimalnih mjesta. Takva dvostruka provjera standardna je praksa kada god ručno izvodite gradijent.

## Treniranje: cijela petlja

Nasumično inicijaliziramo $w, b$ (ovaj put bez domišljate heuristike), a zatim ponavljamo: prolaz unaprijed, računanje gubitka, prolaz unatrag, mali korak *suprotno* gradijentu (gradijentni *spust*, jer gradijent pokazuje prema porastu gubitka).

In [ ]:
def train(X, y, n_epochs=500, lr=0.5, seed=0):
    rng_local = np.random.default_rng(seed)
    w = rng_local.normal(size=X.shape[1]) * 0.1
    b = 0.0
    losses = []
    for _ in range(n_epochs):
        p, _ = forward(X, w, b)
        losses.append(bce_loss(p, y))
        grad_w, grad_b = backward(X, p, y)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b, losses

w_learned, b_learned, losses = train(X, y)
final_pred = (sigmoid(X @ w_learned + b_learned) > 0.5).astype(float)
print(f'learned w = {np.round(w_learned, 3)}, b = {b_learned:.3f}')
print(f'final accuracy: {(final_pred == y).mean():.1%}')

plt.plot(losses)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss')
plt.show()

### Usporedba s ručno odabranim smjerom iz lekcije 30

In [ ]:
w_handpicked = class_b.mean(axis=0) - class_a.mean(axis=0)
w_handpicked /= np.linalg.norm(w_handpicked)
w_learned_normalized = w_learned / np.linalg.norm(w_learned)

cos_angle = w_handpicked @ w_learned_normalized
print(f'hand-picked direction (Lesson 30): {np.round(w_handpicked, 3)}')
print(f'learned direction (this lesson):   {np.round(w_learned_normalized, 3)}')
print(f'angle between them: {np.degrees(np.arccos(np.clip(cos_angle, -1, 1))):.1f} degrees')

Gradijentni spust, počevši samo od nasumičnog šuma, ponovno otkriva gotovo isti smjer koji je čovjek odabrao razmišljanjem o srednjim vrijednostima klasa &mdash; ohrabrujuće, ali ujedno i naznaka ograničenja jednog neurona: može samo *ponovno otkriti* ono što jedna linearna projekcija može postići.

## Treniranje na nerješivom problemu

Lekcija 30 pokazala je da *nijedna* linearna projekcija ne razdvaja prsten od diska koji okružuje &mdash; najbolje što je iscrpno pretraživanje smjerova i pragova pronašlo bila je točnost od 74 %. Što se događa kada isti problem pokušava riješiti gradijentni spust umjesto pretraživanja grubom silom?

In [ ]:
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))
X_ring = np.vstack([inner, outer])
y_ring = np.concatenate([np.zeros(60), np.ones(60)])

w_ring, b_ring, losses_ring = train(X_ring, y_ring, n_epochs=2000)
pred_ring = (sigmoid(X_ring @ w_ring + b_ring) > 0.5).astype(float)
print(f'learned w = {np.round(w_ring, 4)} (norm={np.linalg.norm(w_ring):.4f}), b = {b_ring:.4f}')
print(f'trained single-neuron accuracy: {(pred_ring == y_ring).mean():.1%}')
print(f'(Lesson 30\'s exhaustive-search ceiling for ANY linear separator was 74%)')

Gradijentni spust ovdje zapravo postiže *lošiji* rezultat od pretraživanja grubom silom &mdash; konvergira prema vektoru težina blizu nule i točnosti približno na razini slučajnog pogađanja, umjesto da pronađe mali, asimetrični luk kojim je pretraživanje grubom silom postiglo 74 %. Skup podataka (približno) je simetričan oko ishodišta, pa se *prosječni* utjecaji svih točaka za treniranje na gradijent gotovo poništavaju u svakom smjeru, a optimizator se smiruje blizu $w\approx 0$ umjesto da traži asimetrično rubno rješenje. Drukčiji način neuspjeha, ista temeljna činjenica: jedna linearna projekcija ne može riješiti ovaj problem, neovisno o načinu pronalaženja. Lekcija 32 to rješava tako da modelu daje više projekcija s kojima može raditi, umjesto pametnijeg optimizatora.

### Zadaci

1. Povećajte `lr` u `train` znatno iznad razumne vrijednosti (npr. `lr=20`) na skupu podataka s dvjema skupinama točaka. Što se događa s krivuljom gubitka i kako je to povezano s time da veličina koraka preskače zakrivljenost površine gubitka?
2. Ponovno trenirajte na skupu podataka s dvjema skupinama točaka uz nekoliko različitih početnih vrijednosti generatora slučajnih brojeva. Završava li naučeni smjer uvijek blizu ručno odabranog iz lekcije 30 ili znatno varira? Što to govori o broju dobrih rješenja za dobro razdvojen skup podataka?
3. Učinite vanjski prsten asimetričnim umjesto punog kruga (npr. `theta_out = rng.uniform(0, np.pi, 60)`, tako da vanjska klasa zauzima samo polovicu prstena). Ponovno trenirajte jedan neuron. Mijenja li to stvarno narušavanje simetrije završnu točnost primjetno u usporedbi s rezultatom od oko 50 % na punom prstenu i podupire li gornje objašnjenje (da je simetričan utjecaj *punog* prstena na gradijent, a ne načelna nemogućnost učenja, zaustavio treniranje blizu $w \approx 0$)? (Napomena: samo pomicanje obiju klasa za isti pomak *ne bi* narušilo tu simetriju, jer član pomaka $b$ može bez poteškoća apsorbirati svaku zajedničku translaciju.)